### Data Cleaning Unit Tests

Pytest tests for `clean_name`, `clean_phone` and other fucntions from the Data Load Modules notebook. 

In [0]:
%run ./DataLoadMethods

#### Loading Raw Data

Raw Customer Data - `load_raw_cust_data`

Raw Products Data - `load_raw_product_data`

Raw Orders Data - `load_raw_orders_data`

#### Enriched Data Loads

Cleaning Customers Data - `clean_name`, `clean_phone` and `load_enriched_cust_data`

Cleaning Product Data - `load_enriched_product_data`

Cleaning Orders Data - `clean_raw_orders_data` and `load_enriched_orders_data`

#### Curated Dataset

--------------- End -------------------

In [0]:
import pytest

def apply_fn(fn, col_name, values):
    """Apply a Column-expression function to a list of raw values and return the outputs."""
    df = spark.createDataFrame([(v,) for v in values], f"{col_name} string")
    return [r[0] for r in df.select(fn(col_name).alias("out")).collect()]

def test_apostrophe_preserved_in_surname():
    """Apostrophe between uppercase letters is genuine (O'Rourke) and must survive."""
    out = apply_fn(clean_name, "customer_name", ["Mary O'Rourke", "Russell D'Ascenzo"])
    assert out == ["Mary O'Rourke", "Russell D'Ascenzo"], f"{out}"

def test_removes_junk_and_inner_spaces():
    """Junk runs are deleted together with their surrounding spaces."""
    out = apply_fn(clean_name, "customer_name", ["Ad.       ..am Hart", "Tam&^*ara Willing___)ham"])
    assert out == ["Adam Hart", "Tamara Willingham"], f"{out}"

def test_removes_underscores():
    """Underscores are special characters and must not survive."""
    out = apply_fn(clean_name, "customer_name", ["Matthew Willing___ham"])
    assert out == ["Matthew Willingham"], f"{out}"

def test_removes_digits_inside_words():
    """Digits inside a word are removed without leaving a space behind."""
    out = apply_fn(clean_name, "customer_name", ["Victo 3456ria Brennan", "Ben Pet 987 -=erman"])
    assert out == ["Victoria Brennan", "Ben Peterman"], f"{out}"

def test_joins_lowercase_words_split_by_space():
    """A space between two lowercase letters is corruption, not a name separator."""
    out = apply_fn(clean_name, "customer_name", ["Kat rina Bavinge"])
    assert out == ["Katrina Bavinge"], f"{out}"

def test_joins_lowercase_across_junk_apostrophe():
    """Apostrophe between two lowercase letters (after junk removal) is removed."""
    out = apply_fn(clean_name, "customer_name", ["   Kristi;'[]na Nunn"])
    assert out == ["Kristina Nunn"], f"{out}"

def test_keeps_word_separating_space():
    """The single space between first and last name must survive junk removal."""
    out = apply_fn(clean_name, "customer_name", ["Gary567 Hansen"])
    assert out == ["Gary Hansen"], f"{out}"

def test_keeps_space_after_uppercase():
    """Only lowercase-lowercase gaps are joined; 'B ecky' (uppercase B) is left alone."""
    out = apply_fn(clean_name, "customer_name", ["B ecky Martin"])
    assert out == ["B ecky Martin"], f"{out}"

def test_strips_leading_junk_apostrophes():
    """Junk punctuation at the start of a name is removed (apostrophes included)."""
    out = apply_fn(clean_name, "customer_name", ["[]-=;\"Becky Pak"])
    assert out == ["Becky Pak"], f"{out}"

def test_preserves_unicode_letters():
    """Non-ASCII letters are letters, not special characters."""
    out = apply_fn(clean_name, "customer_name", ["Peter Bühler"])
    assert out == ["Peter Bühler"], f"{out}"

def test_null_name_stays_null():
    """A missing name stays null rather than becoming an empty string."""
    out = apply_fn(clean_name, "customer_name", [None])
    assert out == [None], f"{out}"

# ---------------- clean_phone tests ----------------

def test_removes_extension_and_specials():
    """Extension and punctuation are stripped; a valid 10-digit number gets the +1 prefix."""
    out = apply_fn(clean_phone, "phone", ["421.580.0902x9815"])
    assert out == ["+14215800902"], f"{out}"

def test_strips_001_country_code():
    """A leading 001 country code is removed before the 10-digit check."""
    out = apply_fn(clean_phone, "phone", ["001-542-415-0246x314"])
    assert out == ["+15424150246"], f"{out}"

def test_keeps_plain_10_digit():
    """An already-clean 10-digit number is kept."""
    out = apply_fn(clean_phone, "phone", ["7185624866"])
    assert out == ["+17185624866"], f"{out}"

def test_short_or_invalid_phone_is_null():
    """Anything shorter than 10 digits (or with no digits at all) becomes null."""
    out = apply_fn(clean_phone, "phone", ["-6181", "#ERROR!"])
    assert out == [None, None], f"{out}"

def test_null_phone_is_null():
    """A missing phone stays null."""
    out = apply_fn(clean_phone, "phone", [None])
    assert out == [None], f"{out}"

In [0]:
# ---------------- schema tests: enriched & curated tables ----------------

def assert_schema(table, expected_columns):
    """
    Assert that `table` exists with exactly the expected columns, in order,
    each with the expected type. `expected_columns` is a list of
    (column_name, type) pairs using Spark's simpleString type names.
    """
    actual = [(f.name, f.dataType.simpleString()) for f in spark.table(table).schema.fields]
    assert actual == expected_columns, f"{table}\n  expected: {expected_columns}\n  actual:   {actual}"

def test_enriched_customers_schema():
    """enriched.customers: 11 cleaned customer columns, all strings."""
    assert_schema("enriched.customers", [
        ("customer_id", "string"), ("customer_name", "string"), ("email", "string"),
        ("phone", "string"), ("address", "string"), ("segment", "string"),
        ("country", "string"), ("city", "string"), ("state", "string"),
        ("postal_code", "string"), ("region", "string"),
    ])

def test_enriched_products_schema():
    """enriched.products: product attributes with price cast to decimal(10,2)."""
    assert_schema("enriched.products", [
        ("product_id", "string"), ("category", "string"), ("sub_category", "string"),
        ("product_name", "string"), ("state", "string"), ("price_per_product", "decimal(10,2)"),
    ])

def test_enriched_orders_schema():
    """enriched.orders: dates parsed to date, numerics to decimal/int."""
    assert_schema("enriched.orders", [
        ("order_id", "string"), ("customer_id", "string"), ("product_id", "string"),
        ("order_date", "date"), ("ship_date", "date"), ("ship_mode", "string"),
        ("discount", "decimal(10,2)"), ("quantity", "int"),
        ("price", "decimal(10,2)"), ("profit", "decimal(10,2)"), ("row_id", "int"),
    ])

def test_curated_orders_schema():
    """curated.orders: joined fact table with order_year and profit measure."""
    assert_schema("curated.orders", [
        ("order_id", "string"), ("customer_name", "string"), ("country", "string"),
        ("category", "string"), ("sub_category", "string"),
        ("order_year", "int"), ("profit", "decimal(10,2)"),
    ])

def test_curated_profit_aggregates_schema():
    """curated.profit_aggregates: yearly per customer/category profit rollup."""
    assert_schema("curated.profit_aggregates", [
        ("year", "int"), ("category", "string"), ("sub_category", "string"),
        ("customer_name", "string"), ("profit", "double"),
    ])

In [0]:
# ---------------- duplicate-key tests: enriched tables ----------------

def assert_unique_key(table, key_cols):
    """
    Assert that `table` contains no duplicate rows for the given business key.
    `key_cols` is a single column name or a list of column names.
    """
    keys = key_cols if isinstance(key_cols, list) else [key_cols]
    dupes = (
        spark.table(table)
        .groupBy(*keys)
        .count()
        .filter("count > 1")
    )
    examples = dupes.limit(5).collect()
    assert not examples, f"{table}: duplicate keys {keys} -> {examples}"

def test_enriched_customers_no_duplicate_keys():
    """enriched.customers: customer_id is the unique business key."""
    assert_unique_key("enriched.customers", "customer_id")

def test_enriched_products_no_duplicate_keys():
    """enriched.products: product_id is the unique business key."""
    assert_unique_key("enriched.products", "product_id")

def test_enriched_orders_no_duplicate_keys():
    """enriched.orders: (order_id, product_id) identifies a single order line."""
    assert_unique_key("enriched.orders", ["order_id", "product_id"])

In [0]:
# Run every test_* function defined above, grouped by the method under test,
# and print a pytest-style summary per method.
import inspect
import re

def method_under_test(fn):
    """Return the group a test belongs to, parsed from the test's source."""
    src = inspect.getsource(fn)
    m = re.search(r"apply_fn\(\s*(\w+)", src)
    if m:
        return m.group(1)
    if re.search(r'assert_schema\(\s*"', src) or re.search(r'spark\.table\(\s*"', src):
        return "schema_checks"
    # duplicate-key tests use the assert_unique_key helper
    if re.search(r'assert_unique_key\(\s*"', src):
        return "duplicate_checks"
    return "other"

# group the tests by the method they exercise
tests = sorted(n for n, o in globals().items() if n.startswith("test_") and callable(o))
groups = {}
for name in tests:
    groups.setdefault(method_under_test(globals()[name]), []).append(name)

total_passed = total_failed = 0
for method in sorted(groups):
    print(f"\n--- {method} ({len(groups[method])} tests) ---")
    passed = failed = 0
    for name in groups[method]:
        try:
            globals()[name]()
            print(f"PASSED  {name}")
            passed += 1
        except AssertionError as e:
            print(f"FAILED  {name}: {e}")
            failed += 1
        except Exception as e:
            print(f"ERROR   {name}: {type(e).__name__}: {e}")
            failed += 1
    print(f"-> {method}: {passed} passed, {failed} failed")
    total_passed += passed
    total_failed += failed

print("=" * 50)
print(f"TOTAL: {total_passed} passed, {total_failed} failed, {len(tests)} total")


--- clean_name (11 tests) ---
PASSED  test_apostrophe_preserved_in_surname
PASSED  test_joins_lowercase_across_junk_apostrophe
PASSED  test_joins_lowercase_words_split_by_space
PASSED  test_keeps_space_after_uppercase
PASSED  test_keeps_word_separating_space
PASSED  test_null_name_stays_null
PASSED  test_preserves_unicode_letters
PASSED  test_removes_digits_inside_words
PASSED  test_removes_junk_and_inner_spaces
PASSED  test_removes_underscores
PASSED  test_strips_leading_junk_apostrophes
-> clean_name: 11 passed, 0 failed

--- clean_phone (5 tests) ---
PASSED  test_keeps_plain_10_digit
PASSED  test_null_phone_is_null
PASSED  test_removes_extension_and_specials
PASSED  test_short_or_invalid_phone_is_null
PASSED  test_strips_001_country_code
-> clean_phone: 5 passed, 0 failed

--- duplicate_checks (3 tests) ---
PASSED  test_enriched_customers_no_duplicate_keys
PASSED  test_enriched_orders_no_duplicate_keys
PASSED  test_enriched_products_no_duplicate_keys
-> duplicate_checks: 3 passed, 

In [0]:

import sys
import pytest

sys.dont_write_bytecode = True

exit_code = pytest.main([
    "-v", "--no-header",
    "-p", "no:cacheprovider",                  
    "-o", "python_files=Test_*.py",            
    "/Workspace/PEI Assignment/Test_DataLoadMethodsV2.py",
])
print(f"\npytest exit code: {exit_code} (0 = all passed)")

============================= test session starts ==============================
collecting ... collected 29 items

Test_DataLoadMethodsV2.py::test_clean_name[Mary O'Rourke-Mary O'Rourke] PASSED [  3%]
Test_DataLoadMethodsV2.py::test_clean_name[Russell D'Ascenzo-Russell D'Ascenzo] PASSED [  6%]
Test_DataLoadMethodsV2.py::test_clean_name[  Mary   O'Rourke 42 -Mary O'Rourke] PASSED [ 10%]
Test_DataLoadMethodsV2.py::test_clean_name[Matthew Willing___ham-Matthew Willingham] PASSED [ 13%]
Test_DataLoadMethodsV2.py::test_clean_name[Ad.       ..am Hart-Adam Hart] PASSED [ 17%]
Test_DataLoadMethodsV2.py::test_clean_name[Tam&^*ara Willing___)ham-Tamara Willingham] PASSED [ 20%]
Test_DataLoadMethodsV2.py::test_clean_name[Victo 3456ria Brennan-Victoria Brennan] PASSED [ 24%]
Test_DataLoadMethodsV2.py::test_clean_name[Ben Pet 987 -=erman-Ben Peterman] PASSED [ 27%]
Test_DataLoadMethodsV2.py::test_clean_name[Kat rina Bavinge-Katrina Bavinge] PASSED [ 31%]
Test_DataLoadMethodsV2.py::test_clean_name[